## Microgrid optimization with an algebraic approach using JuMP with H2 approach 

**H2 optimization** notebook take the previous model with H2 but try to simplify it replacing the sizing storage of H2 tanks in the model by a cost reducing the dimension of the problem. 

To do so, we will compare this new model with the previous and compare the performance regarding computation time and the LCOE resulting

In [1]:
using Microgrids
using JuMP
using HiGHS
using DataFrames
using CSV

include("../data/Microgrid_model_data.jl")
include("economics.jl")
include("simulate.jl")
include("create_microgrid.jl")
include("study.jl")
include("mg_jump.jl")

using Printf
using Plots

loading times series from Ouessant_data_2016.csv...


In [2]:
import Pkg
Pkg.status("Microgrids")

Status `~/.julia/environments/v1.12/Project.toml`
  [bd581358] Microgrids v0.10.5 `~/Documents/Centrale_Supelec/Projet_recherche/Microgrids.jl`


## Load Microgrid project data

In [3]:
const tseries = load_microgrid_tseries();

Pload_max = maximum(tseries.Pload) # kW
Pload = tseries.Pload
power_rated_gen_max = 0
power_rated_fc_max = 1.2 * Pload_max
power_rated_ele_max = 1.2* Pload_max
power_rated_hb_max = 0
energy_rated_sto_max = 10.0 * Pload_max
H2_sto_max = 10.0 * Pload_max
power_rated_pv_max = 1.0 * Pload_max
power_rated_wind_max = 5.0 * Pload_max

loading times series from Ouessant_data_2016.csv...


8535.0

Let's first find the min(LCOE) for a range of H2 prices, **price** that will stay constant for the rest of the notebook

In [9]:
H2_price_range = 0:1:10;

In [10]:
results = Study(H2_price_range, true)
df_H2, df_Q = dataframe_result(results, H2_price_range, true)

 75.375499 seconds (3.42 M allocations: 206.079 MiB, 0.72% gc time)
 71.875127 seconds (3.42 M allocations: 206.346 MiB, 0.76% gc time)
 72.297814 seconds (3.42 M allocations: 206.346 MiB, 0.62% gc time)
 75.496418 seconds (3.42 M allocations: 206.346 MiB, 0.04% gc time)
 69.515342 seconds (3.42 M allocations: 206.346 MiB, 0.06% gc time)
 44.540319 seconds (3.42 M allocations: 206.346 MiB, 0.23% gc time)
 61.023725 seconds (3.42 M allocations: 206.346 MiB, 0.86% gc time)
 40.888442 seconds (3.42 M allocations: 206.346 MiB, 1.34% gc time)
 58.146263 seconds (3.42 M allocations: 206.346 MiB, 0.87% gc time)
 77.505913 seconds (3.42 M allocations: 206.346 MiB, 0.04% gc time)
 74.699448 seconds (3.42 M allocations: 206.346 MiB, 0.06% gc time)


(11×10 DataFrame
 Row │ H2_price  LCOE     real_LCOE  power_rated_fc  power_rated_ele  energy_r ⋯
     │ Int64     Float64  Float64    Float64         Float64          Float64  ⋯
─────┼──────────────────────────────────────────────────────────────────────────
   1 │        0  201.838    211.715          1232.5          1289.85           ⋯
   2 │        1  201.838    211.715          1232.5          1289.85
   3 │        2  201.838    211.715          1232.5          1289.85
   4 │        3  201.838    211.715          1232.5          1289.85
   5 │        4  201.838    211.715          1232.5          1289.85           ⋯
   6 │        5  201.838    211.715          1232.5          1289.85
   7 │        6  201.838    211.715          1232.5          1289.85
   8 │        7  201.838    211.715          1232.5          1289.85
   9 │        8  201.838    211.715          1232.5          1289.85           ⋯
  10 │        9  201.838    211.715          1232.5          1289.85
  11 │       1

We find the minimum of this dataframe

In [6]:
id = argmin(df_H2.real_LCOE)
best_H2_price = df_H2.H2_price[id]

1000

Now that we have it we display the power rated corresponding

In [7]:
best_row = df_H2[id, :]

Row,H2_price,LCOE,real_LCOE,power_rated_fc,power_rated_ele,energy_rated_sto,power_rated_pv,power_rated_wind,status,cost_Q
,Int64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,String,Float64
1,1000,201.838,211.715,1232.5,1289.85,2512.4,1707.0,2215.92,OPTIMAL,9.87684


Considering this results we proceed to evaluate with the original H2 model

In [8]:
results = Study(H2_price_range, false)
df = dataframe_result(results, H2_price_range, false)

  8.180210 seconds (2.78 M allocations: 174.293 MiB, 1.11% gc time)


Row,LCOE,power_rated_fc,power_rated_ele,energy_rated_sto,LoH_rated,power_rated_pv,power_rated_wind,status
,Float64,Float64,Float64,Float64,Float64,Float64,Float64,String
1,47.4093,1607.0,0.0,105.0,0.0,0.0,0.0,OPTIMAL


In [12]:
export_to_csv(df_H2)

ArgumentError: ArgumentError: column name :header not found in the data frame

We see that the LCOE of the model is ... compared to the approximate model.

